# SDG / ODD Classification Notebook

This notebook explores multiple Natural Language Processing (NLP) approaches for Sustainable Development Goal (SDG / ODD) classification.

The objective is to automatically predict which SDGs are targeted by a startup description, Business Model Canvas (BMC), or sustainability-related text.

The notebook compares:
- DistilBERT fine-tuning
- RoBERTa fine-tuning
- Zero-shot classification using BART

The workflow includes:
1. Data loading and preprocessing
2. Dataset preparation
3. Transformer-based model training
4. Evaluation using Accuracy and Macro F1-score
5. Prediction on startup and BMC examples
6. Exporting trained models and results


In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy matplotlib

## 1. Imports and Environment Setup

This section imports all required libraries for:
- data manipulation
- preprocessing
- visualization
- deep learning
- transformer models
- evaluation metrics

The notebook mainly relies on:
- PyTorch
- HuggingFace Transformers
- scikit-learn
- pandas and numpy


In [2]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

## 2. Loading the SDG Dataset

In this section, the SDG dataset is loaded into memory.

The dataset contains:
- textual descriptions
- startup information
- SDG labels

This data will later be used to train and evaluate transformer models for SDG classification.


In [3]:
# ============================================================
# 2. LOAD OSDG DATASET
# ============================================================

DATA_PATH = "/kaggle/input/datasets/fatmadorgham/sdg-dataset/osdg-community-data-v2024-01-01.csv"

df = pd.read_csv(DATA_PATH, sep="\t", encoding="utf-8")

print(df.shape)
df.head()

(42635, 7)


,doi,text_id,text,sdg,labels_negative,labels_positive,agreement
0,10.6027/9789289342698-7-en,00021941702cd84171ff33962197ca1f,"From a gender perspective, Paulgaard points ou...",5,1,8,0.777778
1,10.18356/eca72908-en,00028349a7f9b2485ff344ae44ccfd6b,Labour legislation regulates maximum working h...,11,2,1,0.333333
2,10.1787/9789264289062-4-en,0004eb64f96e1620cd852603d9cbe4d4,The average figure also masks large difference...,3,1,8,0.777778
3,10.1787/3726edff-en,0005d3e8b213d9e2cb967666e1aca2e9,Applied research is directed “primarily toward...,9,3,6,0.333333
4,10.1787/5k9b7bn5qzvd-en,0006a887475ccfa5a7f5f51d4ac83d02,The extent to which they are akin to corruptio...,3,1,2,0.333333


## 3. Data Cleaning and Preprocessing

This step prepares the dataset before training.

Typical preprocessing operations include:
- removing missing values
- cleaning unnecessary characters
- normalizing text
- formatting labels

Good preprocessing improves model stability and overall classification performance.


In [4]:
# ============================================================
# 3. BASIC CLEANING
# ============================================================

df = df.copy()

df["text"] = df["text"].fillna("").astype(str)
df["sdg"] = df["sdg"].astype(int)

# Keep useful columns only
df = df[["text", "sdg", "labels_negative", "labels_positive", "agreement"]]

# Remove empty texts
df = df[df["text"].str.strip().str.len() > 20]

# Keep high-confidence labels
df = df[df["agreement"] >= 0.6]

# Convert SDG 1-17 to label 0-16
df["label"] = df["sdg"] - 1

print(df.shape)
print(df["sdg"].value_counts().sort_index())
df.head()

(24797, 6)
sdg
1     1333
2      972
3     1909
4     2509
5     2808
6     1528
7     1999
8      925
9     1470
10     949
11    1322
12     536
13    1173
14     813
15    1264
16    3287
Name: count, dtype: int64


,text,sdg,labels_negative,labels_positive,agreement,label
0,"From a gender perspective, Paulgaard points ou...",5,1,8,0.777778,4
2,The average figure also masks large difference...,3,1,8,0.777778,2
8,The Israel Oceanographic and Limnological Rese...,6,0,3,1.000000,5
9,Previous chapters have discussed ways to make ...,2,0,3,1.000000,1
11,The “War on Terror” and the Framework of Inter...,16,0,7,1.000000,15


## 4. Train / Validation / Test Split

The dataset is divided into:
- Training set
- Validation set
- Test set

Purpose of each split:
- Training set → used to train the model
- Validation set → used for hyperparameter tuning and early stopping
- Test set → used for final unbiased evaluation

This separation helps avoid overfitting and data leakage.


In [5]:
# ============================================================
# 5. TRAIN / VAL / TEST SPLIT
# ============================================================

df_train, df_temp = train_test_split(
    df,
    test_size=0.3,
    random_state=SEED,
    stratify=df["label"]
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=df_temp["label"]
)

print("Train:", df_train.shape)
print("Val:", df_val.shape)
print("Test:", df_test.shape)

Train: (17357, 6)
Val: (3720, 6)
Test: (3720, 6)


## 5. Tokenization and Dataset Preparation

Transformer models cannot directly process raw text.

This section:
- tokenizes the text using a pretrained tokenizer
- converts text into token IDs
- generates attention masks
- prepares PyTorch datasets and dataloaders

These tensors are the actual inputs sent to the transformer model.


In [6]:
# ============================================================
# 6. TOKENIZER + DATASET CLASS
# ============================================================

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SDGDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = SDGDataset(df_train["text"], df_train["label"], tokenizer)
val_dataset = SDGDataset(df_val["text"], df_val["label"], tokenizer)
test_dataset = SDGDataset(df_test["text"], df_test["label"], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 6. DistilBERT Model Architecture

DistilBERT is a lightweight version of BERT based on the Transformer architecture.

Main characteristics:
- faster than BERT
- fewer parameters
- lower memory usage
- strong NLP performance

The model contains:
- embedding layers
- transformer encoder layers
- self-attention mechanisms
- a classification head added for SDG prediction


In [7]:
# ============================================================
# 7. MODEL
# ============================================================

sdg_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=17
)

sdg_model.to(DEVICE)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


## 7. Training Configuration

This section defines:
- optimizer
- learning rate
- scheduler
- loss function
- device configuration (CPU/GPU)

These hyperparameters control how the model learns during training.


In [8]:
# ============================================================
# 8. TRAINING SETUP
# ============================================================

EPOCHS = 3
LR = 2e-5

optimizer = torch.optim.AdamW(sdg_model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

## 8. Evaluation Function

An evaluation function is created to measure model performance during training and testing.

Main metrics:
- Accuracy
- Macro F1-score

Macro F1-score is especially important for imbalanced classification problems such as SDG prediction.


In [9]:
# ============================================================
# 9. EVALUATION FUNCTION
# ============================================================

def evaluate(model, loader):
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            preds = torch.argmax(logits, dim=1)

            total_loss += loss.item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return total_loss / len(loader), acc, macro_f1

In [10]:
import os

EXPORT_DIR = "/kaggle/working/sdg_outputs"
os.makedirs(EXPORT_DIR, exist_ok=True)

## 9. DistilBERT Training Loop

This section performs the complete training process.

For each epoch:
1. The model learns on the training set
2. Validation metrics are computed
3. Loss and F1-score are monitored
4. The best model is saved

This helps track model convergence and detect overfitting.


In [11]:
# ============================================================
# 10. TRAINING LOOP
# ============================================================

best_val_f1 = 0
best_model_path = f"{EXPORT_DIR}/best_sdg_distilbert"

for epoch in range(EPOCHS):
    sdg_model.train()
    total_train_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = sdg_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_train_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(sdg_model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

    train_loss = total_train_loss / len(train_loader)
    val_loss, val_acc, val_f1 = evaluate(sdg_model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train loss: {train_loss:.4f}")
    print(f"Val loss: {val_loss:.4f}")
    print(f"Val acc: {val_acc:.4f}")
    print(f"Val macro F1: {val_f1:.4f}")
    print("-" * 50)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        sdg_model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        print("Best model saved!")

Epoch 1/3
Train loss: 1.1326
Val loss: 0.5839
Val acc: 0.8495
Val macro F1: 0.8184
--------------------------------------------------


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved!
Epoch 2/3
Train loss: 0.4677
Val loss: 0.5166
Val acc: 0.8680
Val macro F1: 0.8404
--------------------------------------------------


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved!
Epoch 3/3
Train loss: 0.3453
Val loss: 0.5156
Val acc: 0.8720
Val macro F1: 0.8445
--------------------------------------------------


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved!


## 10. Final Test Evaluation

After training, the best model is evaluated on the test set.

This provides the final performance metrics on unseen data and measures the model's generalization capability.


In [12]:
# ============================================================
# 11. FINAL TEST EVALUATION
# ============================================================

sdg_model = AutoModelForSequenceClassification.from_pretrained(best_model_path)
sdg_model.to(DEVICE)

test_loss, test_acc, test_f1 = evaluate(sdg_model, test_loader)

print("TEST RESULTS")
print("Loss:", test_loss)
print("Accuracy:", test_acc)
print("Macro F1:", test_f1)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

TEST RESULTS
Loss: 0.5049775207781384
Accuracy: 0.8723118279569892
Macro F1: 0.8487985333133079


## 11. Test Predictions and Analysis

This section recomputes predictions on the test set in order to:
- analyze classification behavior
- generate prediction reports
- inspect errors
- build confusion matrices and metrics


In [13]:
# ============================================================
# RECOMPUTE PREDICTIONS FOR TEST SET
# ============================================================

sdg_model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = sdg_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [14]:
from sklearn.metrics import classification_report

target_names = [f"SDG {i}" for i in range(1, 18)]

print(classification_report(
    all_labels,
    all_preds,
    labels=list(range(17)),
    target_names=target_names,
    zero_division=0
))

              precision    recall  f1-score   support

       SDG 1       0.77      0.77      0.77       200
       SDG 2       0.90      0.81      0.85       146
       SDG 3       0.92      0.94      0.93       286
       SDG 4       0.93      0.93      0.93       376
       SDG 5       0.85      0.88      0.87       421
       SDG 6       0.86      0.88      0.87       229
       SDG 7       0.89      0.88      0.89       300
       SDG 8       0.65      0.68      0.66       139
       SDG 9       0.83      0.89      0.86       221
      SDG 10       0.73      0.63      0.67       143
      SDG 11       0.81      0.84      0.83       198
      SDG 12       0.78      0.80      0.79        81
      SDG 13       0.83      0.84      0.84       176
      SDG 14       0.96      0.94      0.95       122
      SDG 15       0.93      0.88      0.91       189
      SDG 16       0.98      0.97      0.98       493
      SDG 17       0.00      0.00      0.00         0

    accuracy              

## 12. SDG Prediction Function

A reusable prediction function is created.

The function:
- receives a text input
- tokenizes the text
- runs inference through the model
- returns predicted SDGs with confidence scores

This simulates real-world SDG classification usage.


In [15]:
# ============================================================
# 13. PREDICTION FUNCTION
# ============================================================

sdg_names = {
    1: "No Poverty",
    2: "Zero Hunger",
    3: "Good Health and Well-being",
    4: "Quality Education",
    5: "Gender Equality",
    6: "Clean Water and Sanitation",
    7: "Affordable and Clean Energy",
    8: "Decent Work and Economic Growth",
    9: "Industry, Innovation and Infrastructure",
    10: "Reduced Inequalities",
    11: "Sustainable Cities and Communities",
    12: "Responsible Consumption and Production",
    13: "Climate Action",
    14: "Life Below Water",
    15: "Life on Land",
    16: "Peace, Justice and Strong Institutions",
    17: "Partnerships for the Goals"
}

def predict_sdg(text, top_k=3):
    sdg_model.eval()

    encoded = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)

    with torch.no_grad():
        outputs = sdg_model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]

    top_indices = probs.argsort()[-top_k:][::-1]

    results = []
    for idx in top_indices:
        sdg_number = idx + 1
        results.append({
            "sdg": sdg_number,
            "name": sdg_names[sdg_number],
            "confidence": float(probs[idx])
        })

    return results

## 13. Testing on Startup and BMC Examples

The trained model is tested on realistic startup and Business Model Canvas examples.

This demonstrates how the model can classify sustainability goals from entrepreneurial and business-related descriptions.


In [16]:
# ============================================================
# 14. TEST ON STARTUP / BMC TEXT
# ============================================================

sample_bmc = """
The startup provides an AI platform that helps companies reduce energy consumption,
optimize logistics, and monitor carbon emissions. It supports small businesses in
making greener operational decisions and reducing waste.
"""

predict_sdg(sample_bmc, top_k=3)

[{'sdg': np.int64(9),
  'name': 'Industry, Innovation and Infrastructure',
  'confidence': 0.8262190818786621},
 {'sdg': np.int64(7),
  'name': 'Affordable and Clean Energy',
  'confidence': 0.04646947234869003},
 {'sdg': np.int64(12),
  'name': 'Responsible Consumption and Production',
  'confidence': 0.030047167092561722}]

## 14. Exporting Results

This section exports:
- trained models
- evaluation results
- prediction outputs
- logs and metrics

The files are compressed for easier download and deployment.


In [17]:
# ============================================================
# 15. ZIP OUTPUTS FOR DOWNLOAD
# ============================================================

!zip -r /kaggle/working/sdg_outputs.zip /kaggle/working/sdg_outputs

  adding: kaggle/working/sdg_outputs/ (stored 0%)
  adding: kaggle/working/sdg_outputs/best_sdg_distilbert/ (stored 0%)
  adding: kaggle/working/sdg_outputs/best_sdg_distilbert/config.json (deflated 62%)
  adding: kaggle/working/sdg_outputs/best_sdg_distilbert/model.safetensors (deflated 8%)
  adding: kaggle/working/sdg_outputs/best_sdg_distilbert/tokenizer.json (deflated 71%)
  adding: kaggle/working/sdg_outputs/best_sdg_distilbert/tokenizer_config.json (deflated 42%)


**PART 2 — RoBERTa (fine-tuning)**

# RoBERTa SDG Classification

This second experiment uses RoBERTa for SDG classification.

RoBERTa is an improved version of BERT trained with:
- larger datasets
- longer training
- optimized masking strategies

The objective is to compare RoBERTa against DistilBERT and evaluate whether it improves SDG classification performance.


In [18]:
# ============================================================
# RoBERTa SDG CLASSIFICATION
# Dataset + Loaders + Training Loop
# ============================================================

import os
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EXPORT_DIR = "/kaggle/working/sdg_outputs"
os.makedirs(EXPORT_DIR, exist_ok=True)

ROBERTA_MODEL_NAME = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL_NAME)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Dataset Class for RoBERTa

A custom dataset class is implemented to:
- preprocess text
- tokenize inputs
- generate tensors
- prepare labels

This class allows efficient batching and training with PyTorch DataLoaders.


In [19]:
# ============================================================
# Dataset Class
# ============================================================

class SDGRobertaDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts.fillna("").astype(str).tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

## DataLoaders for RoBERTa

This section creates:
- training dataloader
- validation dataloader
- test dataloader

DataLoaders efficiently feed batches of tokenized text into the transformer model during training and evaluation.


In [20]:
# ============================================================
# Datasets + Loaders
# df_train, df_val, df_test must already exist
# They must contain columns: "text" and "label"
# label = SDG - 1, so values from 0 to 16
# ============================================================

train_roberta_dataset = SDGRobertaDataset(
    df_train["text"],
    df_train["label"],
    roberta_tokenizer
)

val_roberta_dataset = SDGRobertaDataset(
    df_val["text"],
    df_val["label"],
    roberta_tokenizer
)

test_roberta_dataset = SDGRobertaDataset(
    df_test["text"],
    df_test["label"],
    roberta_tokenizer
)

train_roberta_loader = DataLoader(
    train_roberta_dataset,
    batch_size=8,
    shuffle=True
)

val_roberta_loader = DataLoader(
    val_roberta_dataset,
    batch_size=16,
    shuffle=False
)

test_roberta_loader = DataLoader(
    test_roberta_dataset,
    batch_size=16,
    shuffle=False
)

## RoBERTa Model Definition

The pretrained RoBERTa transformer is loaded and adapted for SDG classification.

A classification head is added on top of the transformer output in order to predict SDG labels.


In [21]:
# ============================================================
# Model
# ============================================================

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_MODEL_NAME,
    num_labels=17
)

roberta_model.to(DEVICE)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

## Evaluation Function for RoBERTa

This evaluation pipeline measures:
- validation accuracy
- macro F1-score
- prediction quality

The same metrics are used across models to ensure fair comparison.


In [22]:
# ============================================================
# Evaluation Function
# ============================================================

def evaluate_roberta(model, loader):
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            total_loss += loss.item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return total_loss / len(loader), acc, macro_f1

## RoBERTa Training Setup

The optimizer, scheduler, learning rate, and training parameters are configured specifically for RoBERTa fine-tuning.


In [23]:
# ============================================================
# Training Setup
# ============================================================

EPOCHS = 3
LR = 2e-5

optimizer = torch.optim.AdamW(roberta_model.parameters(), lr=LR)

total_steps = len(train_roberta_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

best_val_f1 = 0
best_roberta_path = f"{EXPORT_DIR}/best_roberta_sdg"

## RoBERTa Training Loop

This section trains the RoBERTa model across multiple epochs while monitoring validation performance and saving the best checkpoint.


In [ ]:
# ============================================================
# Training Loop
# ============================================================

for epoch in range(EPOCHS):
    roberta_model.train()
    total_train_loss = 0

    for batch in train_roberta_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = roberta_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_train_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(roberta_model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

    train_loss = total_train_loss / len(train_roberta_loader)
    val_loss, val_acc, val_f1 = evaluate_roberta(roberta_model, val_roberta_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train loss: {train_loss:.4f}")
    print(f"Val loss: {val_loss:.4f}")
    print(f"Val acc: {val_acc:.4f}")
    print(f"Val macro F1: {val_f1:.4f}")
    print("-" * 50)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        roberta_model.save_pretrained(best_roberta_path)
        roberta_tokenizer.save_pretrained(best_roberta_path)
        print("Best RoBERTa model saved!")

## Final Evaluation of RoBERTa

The trained RoBERTa model is evaluated on the test dataset to compare its performance against DistilBERT.


In [ ]:
# ============================================================
# Final Test Evaluation
# ============================================================

best_roberta_model = AutoModelForSequenceClassification.from_pretrained(best_roberta_path)
best_roberta_model.to(DEVICE)

test_loss, test_acc, test_f1 = evaluate_roberta(best_roberta_model, test_roberta_loader)

print("RoBERTa TEST RESULTS")
print("Loss:", test_loss)
print("Accuracy:", test_acc)
print("Macro F1:", test_f1)

**PART 2 — Zero-shot with BART**

# Zero-Shot SDG Classification with BART

This final experiment uses zero-shot classification.

Unlike DistilBERT and RoBERTa:
- no fine-tuning is performed
- the model predicts labels directly using natural language inference

The model used is BART-large-MNLI.

This approach is useful when labeled data is limited.


In [ ]:
# ============================================================
# ZERO-SHOT SDG CLASSIFICATION WITH BART
# ============================================================

import torch
from transformers import pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if DEVICE == "cuda" else -1
)

## SDG Labels Definition

The list of SDG labels is defined here.

These labels are used as candidate classes during zero-shot prediction.


In [ ]:
# ============================================================
# SDG LABELS
# ============================================================

sdg_labels = [
    "No Poverty",
    "Zero Hunger",
    "Good Health and Well-being",
    "Quality Education",
    "Gender Equality",
    "Clean Water and Sanitation",
    "Affordable and Clean Energy",
    "Decent Work and Economic Growth",
    "Industry, Innovation and Infrastructure",
    "Reduced Inequalities",
    "Sustainable Cities and Communities",
    "Responsible Consumption and Production",
    "Climate Action",
    "Life Below Water",
    "Life on Land",
    "Peace, Justice and Strong Institutions",
    "Partnerships for the Goals"
]

## Zero-Shot Evaluation Function

This section evaluates the zero-shot model on the SDG dataset using:
- Accuracy
- Macro F1-score

The results can then be compared against fine-tuned transformer models.


In [ ]:
# ============================================================
# ZERO-SHOT EVALUATION FUNCTION
# ============================================================

from sklearn.metrics import accuracy_score, f1_score

def evaluate_zero_shot(df, max_samples=1000):
    """
    Evaluate zero-shot model on a subset of data
    df must contain: text, label (0–16)
    """

    true_labels = []
    pred_labels = []

    # limit samples (VERY IMPORTANT → BART is slow)
    df_sample = df.sample(n=min(max_samples, len(df)), random_state=42)

    for i, row in df_sample.iterrows():
        text = row["text"]
        true_label = row["label"]  # already 0–16

        # get top prediction
        result = zero_shot_classifier(
            text,
            candidate_labels=sdg_labels,
            multi_label=False,  # IMPORTANT for single-label eval
            hypothesis_template="This text is related to {}."
        )

        predicted_label_name = result["labels"][0]

        # convert label name → index
        pred_label = sdg_labels.index(predicted_label_name)

        true_labels.append(true_label)
        pred_labels.append(pred_label)

    acc = accuracy_score(true_labels, pred_labels)
    macro_f1 = f1_score(true_labels, pred_labels, average="macro")

    return acc, macro_f1

In [ ]:
acc, f1 = evaluate_zero_shot(df_test, max_samples=500)

print("ZERO-SHOT RESULTS")
print("Accuracy:", acc)
print("Macro F1:", f1)

## Zero-Shot Prediction Function

A reusable prediction function is implemented for zero-shot SDG classification.

The model compares the input text against all SDG labels and selects the most relevant goals.


In [ ]:
# ============================================================
# ZERO-SHOT PREDICTION FUNCTION
# ============================================================

def predict_sdg_zero_shot(text, top_k=3):
    result = zero_shot_classifier(
        text,
        candidate_labels=sdg_labels,
        multi_label=True,
        hypothesis_template="This text is related to {}."
    )

    labels_scores = list(zip(result["labels"], result["scores"]))
    labels_scores = sorted(labels_scores, key=lambda x: x[1], reverse=True)

    return labels_scores[:top_k]

## Testing the Zero-Shot Model

The zero-shot classifier is tested on startup and Business Model Canvas examples to analyze how well it generalizes without additional training.


In [ ]:
# ============================================================
# TEST ON ONE STARTUP / BMC EXAMPLE
# ============================================================

sample_text = """
The startup develops solar energy solutions for rural areas.
It helps communities access affordable clean energy, reduces carbon emissions,
and supports sustainable infrastructure.
"""

predict_sdg_zero_shot(sample_text, top_k=5)

In [ ]:
bmc_text = """
Customer Segments: small businesses and households.
Value Proposition: an AI platform that helps reduce energy consumption and carbon emissions.
Key Activities: energy monitoring, optimization, and sustainability reporting.
Revenue Streams: monthly subscription.
"""

predict_sdg_zero_shot(bmc_text, top_k=5)